<a href="https://colab.research.google.com/github/Natalyrubin/School-Bullying-Analytics/blob/main/NatalyRubin_FinalProject_Bullying.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Upload Data | op-1 | import from kaggle

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os


path = kagglehub.dataset_download("leomartinelli/bullying-in-schools")
print(os.listdir(path))

df = pd.read_csv(os.path.join(path, "Bullying_2018.csv"))
df.head()

In [ ]:
df = pd.read_csv(os.path.join(path, "Bullying_2018.csv"), sep=';')

In [ ]:
df.head()

*
*
*
*

END | op-1
---


*
*
*
*

# Upload Data | op-2 | import from local

In [ ]:
#from google.colab import files
#files.upload()

In [ ]:
#import pandas as pd
#import numpy as np

#df = pd.read_csv("Bullying_2018.csv")

In [ ]:
#df.head()


In [ ]:
#df = pd.read_csv("Bullying_2018.csv", sep=';')

In [ ]:
#df.head()

*
*
*
*

END | op-2
---


*
*
*
*



---


# Data Overview & Missing Values Analysis

In [ ]:
df.info()
df.describe()
df.sample()

In [ ]:
def count_empty_after_strip(col):
        return col.isna().sum() + col.astype(str).str.strip().eq('').sum()

empty_counts = df.apply(count_empty_after_strip)
print(empty_counts)

In [ ]:
(empty_counts / len(df)).sort_values() * 100

In [ ]:
df = df.apply(lambda col: col.mask(col.astype(str).str.strip() == '', np.nan))

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df[['Were_underweight', 'Were_overweight', 'Were_obese']].notna().all(axis=1).sum()

In [ ]:
df[['Were_underweight', 'Were_overweight', 'Were_obese']].isna().all(axis=1).sum()



---


# Data Cleaning & Preparation

בחינה האם כדאי לעבוד עם הדאטה המלא ולפלטר משקל בשאלות משקל / או לפלטר מראש ולעבוד רק עם מי שענה על שאלות משקל.
על מנת לא להטות את המחקר כתוצאה מההשערה שמי שלא ענה זה כתוצאה מבושה במשקל וככל הנראה משפיע על התוצאה.

In [ ]:
df_analysis = df.copy()

In [ ]:
df_analysis["bullied"] = (
    (df["Bullied_on_school_property_in_past_12_months"] == "Yes") |
    (df["Bullied_not_on_school_property_in_past_12_months"] == "Yes") |
    (df["Cyber_bullied_in_past_12_months"] == "Yes")
).astype(int)

In [ ]:
df_analysis

In [ ]:
df_analysis["bullied"].mean()

In [ ]:
df_analysis["weight_missing"] = df["Were_underweight"].isnull().astype(int)

In [ ]:
pd.crosstab(df_analysis["weight_missing"], df_analysis["bullied"], normalize="index")

לא נמצאה עדות להבדל משמעותי בשיעור הבריונות בין תלמידים שדיווחו על משקל לבין אלו שלא, ולכן נבחר לעבוד עם כלל המדגם. עם זאת, ניתוח משתני המשקל יבוצע על תת-אוכלוסייה בלבד



---


# Setup: Install & Import Libraries

In [ ]:
# System & DB connectivity (Optional / Future use)
#!apt-get install -y libsasl2-dev
#!pip install thrift
#!pip install sasl
#!pip install thrift_sasl
#!pip install sqlalchemy
#!pip install impyla
#!pip install pure-sasl


In [ ]:
# Visualization (Used in EDA)
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning - Classification (Used)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Machine Learning - General (Used)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Machine Learning - Regression (Optional / Future use)
#from sklearn.linear_model import LinearRegression

# Model Evaluation (Used)
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    roc_curve,
    auc
)

# Statistical Analysis (Used)
from scipy.stats import (
    chi2_contingency,
    norm,
    ttest_ind,
    ttest_1samp,
    t,
    pearsonr,
    spearmanr,
    binom
)

# Database Connectivity (Optional / Future use)
#import sqlalchemy
#from impala.dbapi import connect

# Tree Visualization (Optional)
#from sklearn import tree

In [ ]:
df_analysis["Felt_lonely_num"] = df_analysis["Felt_lonely"].map({
    "Never": 0,
    "Rarely": 1,
    "Sometimes": 2,
    "Most of the time": 3,
    "Always": 4
})

df_analysis["Close_friends_num"] = df_analysis["Close_friends"].map({
    "0": 0,
    "1": 1,
    "2": 2,
    "3 or more": 3
})

df_analysis["Sex_num"] = df_analysis["Sex"].map({
    "Male": 0,
    "Female": 1
})

In [ ]:
pd.crosstab(df_analysis["Felt_lonely_num"], df_analysis["bullied"], normalize="index") * 100

In [ ]:
lonely_table = pd.crosstab(df_analysis["Felt_lonely_num"], df_analysis["bullied"])
lonely_table

In [ ]:
lonely_table_pct = pd.crosstab(
    df_analysis["Felt_lonely"],
    df_analysis["bullied"],
    normalize="index"
)

order = ["Never", "Rarely", "Sometimes", "Most of the time", "Always"]
lonely_table_pct = lonely_table_pct.reindex(order)

lonely_table_pct.plot(kind="bar", figsize=(8,5))
plt.title("Bullying Rate by Loneliness Level")
plt.xlabel("Felt Lonely")
plt.ylabel("Percentage")
plt.legend(["Not Bullied", "Bullied"])
plt.show()

In [ ]:
lonely_table_pct[1].plot(marker='o', figsize=(8,5))
plt.title("Bullying Probability by Loneliness Level")
plt.xlabel("Felt Lonely")
plt.ylabel("Probability")
plt.grid()
plt.show()

In [ ]:
sns.heatmap(lonely_table_pct, annot=True, cmap="Reds", fmt=".2f")
plt.title("Bullying Rate Heatmap")
plt.xlabel("Bullied")
plt.ylabel("Felt Lonely")
plt.show()

In [ ]:
chi2, p, dof, expected = chi2_contingency(lonely_table)

lonely_expected_df = pd.DataFrame(expected,
                           index=lonely_table.index,
                           columns=lonely_table.columns)

lonely_expected_df

In [ ]:
print("Chi-Square Test Results:")
print("------------------------")
print(f"Chi2 Statistic: {chi2:.2f}")
print(f"P-value: {p:.5f}")
print(f"Degrees of Freedom: {dof}")

alpha = 0.05
if p < alpha:
    print("\nResult: Significant relationship (reject H0)")
else:
    print("\nResult: No significant relationship (fail to reject H0)")

In [ ]:
df_analysis["Felt_lonely"].value_counts(dropna=False)

In [ ]:
lonely_model_data = df_analysis[["Felt_lonely_num", "bullied"]].dropna()

X = lonely_model_data[["Felt_lonely_num"]]
y = lonely_model_data["bullied"]

model = LogisticRegression()
model.fit(X, y)

In [ ]:
model.coef_


In [ ]:
model.intercept_

In [ ]:
x_vals = np.array([0,1,2,3,4])
probs = model.predict_proba(x_vals.reshape(-1,1))[:,1]

plt.plot(x_vals, probs, marker='o')
plt.xticks(x_vals, ["Never","Rarely","Sometimes","Most","Always"])
plt.ylabel("Probability of Being Bullied")
plt.title("Logistic Regression Effect")
plt.grid()
plt.show()

In [ ]:
spearmanr(
    lonely_model_data["Felt_lonely_num"],
    lonely_model_data["bullied"]
)

In [ ]:
pd.crosstab(df_analysis["Close_friends_num"], df_analysis["bullied"], normalize="index") * 100

In [ ]:
friends_table = pd.crosstab(df_analysis["Close_friends_num"], df_analysis["bullied"])
friends_table

In [ ]:
friends_table_pct = pd.crosstab(
    df_analysis["Close_friends"],
    df_analysis["bullied"],
    normalize="index"
)

friends_order = ["0", "1", "2", "3 or more"]
friends_table_pct = friends_table_pct.reindex(friends_order)

friends_table_pct.plot(kind="bar", figsize=(8,5))
plt.title("Bullying Rate by Number of Close Friends")
plt.xlabel("Number of Close Friends")
plt.ylabel("Percentage")
plt.legend(["Not Bullied", "Bullied"])
plt.show()

In [ ]:
friends_table_pct[1].plot(marker='o', figsize=(8,5))
plt.title("Bullying Probability by Number of Close Friends")
plt.xlabel("Number of Close Friends")
plt.ylabel("Probability")
plt.grid()
plt.show()

In [ ]:
sns.heatmap(friends_table_pct, annot=True, cmap="Reds", fmt=".2f")
plt.title("Bullying Rate Heatmap - Close Friends")
plt.xlabel("Bullied")
plt.ylabel("Number of Close Friends")
plt.show()

In [ ]:
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(friends_table)

friends_expected_df = pd.DataFrame(
    expected,
    index=friends_table.index,
    columns=friends_table.columns
)

friends_expected_df

In [ ]:
print("Chi-Square Test Results:")
print("------------------------")
print(f"Chi2 Statistic: {chi2:.2f}")
print(f"P-value: {p:.5f}")
print(f"Degrees of Freedom: {dof}")

alpha = 0.05
if p < alpha:
    print("\nResult: Significant relationship (reject H0)")
else:
    print("\nResult: No significant relationship (fail to reject H0)")

In [ ]:
df_analysis["Close_friends"].value_counts(dropna=False)

In [ ]:
friends_model_data = df_analysis[["Close_friends_num", "bullied"]].dropna()

X = friends_model_data[["Close_friends_num"]]
y = friends_model_data["bullied"]

friends_model = LogisticRegression()
friends_model.fit(X, y)

friends_model.coef_

In [ ]:
friends_model.intercept_

In [ ]:
x_vals = np.array([0,1,2,3])
probs = friends_model.predict_proba(x_vals.reshape(-1,1))[:,1]

plt.plot(x_vals, probs, marker='o')
plt.xticks(x_vals, ["0","1","2","3+"])
plt.ylabel("Probability of Being Bullied")
plt.title("Logistic Regression Effect - Close Friends")
plt.grid()
plt.show()

In [ ]:
spearmanr(
    friends_model_data["Close_friends_num"],
    friends_model_data["bullied"]
)

In [ ]:
pd.crosstab(df_analysis["Sex_num"], df_analysis["bullied"], normalize="index") * 100

In [ ]:
sex_table = pd.crosstab(df_analysis["Sex_num"], df_analysis["bullied"])
sex_table

In [ ]:
sex_table_pct = pd.crosstab(
    df_analysis["Sex"],
    df_analysis["bullied"],
    normalize="index"
)

sex_table_pct

In [ ]:
sex_table_pct.plot(kind="bar", figsize=(6,4))
plt.title("Bullying Rate by Sex")
plt.xlabel("Sex")
plt.ylabel("Percentage")
plt.legend(["Not Bullied", "Bullied"])
plt.show()

In [ ]:
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(sex_table)

sex_expected_df = pd.DataFrame(
    expected,
    index=sex_table.index,
    columns=sex_table.columns
)

sex_expected_df

In [ ]:
print("Chi-Square Test Results:")
print("------------------------")
print(f"Chi2 Statistic: {chi2:.2f}")
print(f"P-value: {p:.5f}")
print(f"Degrees of Freedom: {dof}")

if p < 0.05:
    print("\nResult: Significant relationship (reject H0)")
else:
    print("\nResult: No significant relationship (fail to reject H0)")

In [ ]:
sex_model_data = df_analysis[["Sex_num", "bullied"]].dropna()

X = sex_model_data[["Sex_num"]]
y = sex_model_data["bullied"]

sex_model = LogisticRegression()
sex_model.fit(X, y)


In [ ]:
x_vals = np.array([0,1])
probs = sex_model.predict_proba(x_vals.reshape(-1,1))[:,1]

plt.plot(x_vals, probs, marker='o')
plt.xticks(x_vals, ["Male","Female"])
plt.ylabel("Probability of Being Bullied")
plt.title("Logistic Regression Effect - Sex")
plt.grid()
plt.show()

In [ ]:
sex_model.coef_

In [ ]:
from statsmodels.stats.proportion import proportions_ztest


count = sex_table[1].values

nobs = sex_table.sum(axis=1).values

stat, pval = proportions_ztest(count, nobs)

print(stat, pval)

In [ ]:
odds = (sex_table[1] / sex_table[0])
odds_ratio = odds[1] / odds[0]
odds_ratio

In [ ]:
cols = [
    "Physically_attacked",
    "Physical_fighting",
    "Miss_school_no_permission",
    "Other_students_kind_and_helpful",
    "Parents_understand_problems",
    "Missed_classes_or_school_without_permission",
    "Custom_Age"
]

for col in cols:
    print(f"\n===== {col} =====")
    print(df_analysis[col].value_counts(dropna=False))

In [ ]:
df_analysis["Physically_attacked_group"] = df_analysis["Physically_attacked"].replace({
    "0 times": 0,
    "1 time": 1,
    "2 or 3 times": 2,
    "4 or 5 times": 3,
    "6 or 7 times": 3,
    "8 or 9 times": 3,
    "10 or 11 times": 3,
    "12 or more times": 3
})

In [ ]:
df_analysis["Physically_attacked_group"].value_counts()

In [ ]:
pd.crosstab(df_analysis["Physically_attacked_group"], df_analysis["bullied"], normalize="index")

In [ ]:
attacked_table = pd.crosstab(df_analysis["Physically_attacked_group"], df_analysis["bullied"])
attacked_table

In [ ]:
attacked_table_pct = pd.crosstab(
    df_analysis["Physically_attacked_group"],
    df_analysis["bullied"],
    normalize="index"
)

attacked_table_pct = attacked_table_pct.sort_index()

attacked_labels = ["0 times", "1 time", "2-3 times", "4+ times"]
attacked_table_pct.index = attacked_labels

attacked_table_pct.plot(kind="bar", figsize=(8,5))
plt.title("Bullying Rate by Physical Attacks")
plt.xlabel("Physical Attacks")
plt.ylabel("Percentage")
plt.legend(["Not Bullied", "Bullied"])
plt.show()

In [ ]:
attacked_table_pct[1].plot(marker='o', figsize=(8,5))
plt.title("Bullying Probability by Physical Attacks")
plt.xlabel("Number of Physical Attacks")
plt.ylabel("Probability")
plt.grid()
plt.show()

In [ ]:
chi2, p, dof, expected = chi2_contingency(attacked_table)

print("Chi-Square Test Results:")
print("------------------------")
print(f"Chi2 Statistic: {chi2:.2f}")
print(f"P-value: {p:.5f}")
print(f"Degrees of Freedom: {dof}")

if p < 0.05:
    print("\nResult: Significant relationship (reject H0)")
else:
    print("\nResult: No significant relationship (fail to reject H0)")

In [ ]:
attacked_expected_df = pd.DataFrame(
    expected,
    index=attacked_table.index,
    columns=attacked_table.columns
)

attacked_expected_df

In [ ]:
attacked_model_data = df_analysis[["Physically_attacked_group", "bullied"]].dropna()

spearmanr(
    attacked_model_data["Physically_attacked_group"],
    attacked_model_data["bullied"]
)

In [ ]:
X = attacked_model_data[["Physically_attacked_group"]]
y = attacked_model_data["bullied"]

attacked_model = LogisticRegression()
attacked_model.fit(X, y)

attacked_model.coef_

In [ ]:
attacked_model.intercept_

In [ ]:
x_vals = np.array([0,1,2,3])
probs = attacked_model.predict_proba(x_vals.reshape(-1,1))[:,1]

plt.plot(x_vals, probs, marker='o')
plt.xticks(x_vals, ["0","1","2-3","4+"])
plt.ylabel("Probability of Being Bullied")
plt.title("Logistic Regression Effect - Physical Attacks")
plt.grid()
plt.show()

In [ ]:
df_analysis["Physical_fighting_group"] = df_analysis["Physical_fighting"].replace({
    "0 times": 0,
    "1 time": 1,
    "2 or 3 times": 2,
    "4 or 5 times": 3,
    "6 or 7 times": 3,
    "8 or 9 times": 3,
    "10 or 11 times": 3,
    "12 or more times": 3
})

In [ ]:
df_analysis["Physical_fighting_group"].value_counts()

In [ ]:
fighting_table_pct = pd.crosstab(
    df_analysis["Physical_fighting_group"],
    df_analysis["bullied"],
    normalize="index"
)

fighting_table_pct = fighting_table_pct.sort_index()

labels = ["0 times", "1 time", "2-3 times", "4+ times"]
fighting_table_pct.index = labels

fighting_table_pct

In [ ]:
fighting_table_pct.plot(kind="bar", figsize=(8,5))
plt.title("Bullying Rate by Physical Fighting")
plt.xlabel("Physical Fighting")
plt.ylabel("Percentage")
plt.legend(["Not Bullied", "Bullied"])
plt.show()

In [ ]:
fighting_table_pct[1].plot(marker='o', figsize=(8,5))
plt.title("Bullying Probability by Physical Fighting")
plt.xlabel("Physical Fighting")
plt.ylabel("Probability")
plt.grid()
plt.show()

In [ ]:
fighting_table = pd.crosstab(
    df_analysis["Physical_fighting_group"],
    df_analysis["bullied"]
)

chi2, p, dof, expected = chi2_contingency(fighting_table)

print("Chi-Square Test Results:")
print("------------------------")
print(f"Chi2 Statistic: {chi2:.2f}")
print(f"P-value: {p:.5f}")
print(f"Degrees of Freedom: {dof}")

if p < 0.05:
    print("\nResult: Significant relationship (reject H0)")
else:
    print("\nResult: No significant relationship (fail to reject H0)")

In [ ]:
fighting_expected_df = pd.DataFrame(
    expected,
    index=fighting_table.index,
    columns=fighting_table.columns
)

fighting_expected_df

In [ ]:
fighting_model_data = df_analysis[["Physical_fighting_group", "bullied"]].dropna()

spearmanr(
    fighting_model_data["Physical_fighting_group"],
    fighting_model_data["bullied"]
)

In [ ]:
X = fighting_model_data[["Physical_fighting_group"]]
y = fighting_model_data["bullied"]

fighting_model = LogisticRegression()
fighting_model.fit(X, y)

fighting_model.coef_

In [ ]:
fighting_model.intercept_

In [ ]:
x_vals = np.array([0,1,2,3])
probs = fighting_model.predict_proba(x_vals.reshape(-1,1))[:,1]

plt.plot(x_vals, probs, marker='o')
plt.xticks(x_vals, ["0","1","2-3","4+"])
plt.ylabel("Probability of Being Bullied")
plt.title("Logistic Regression Effect - Physical Fighting")
plt.grid()
plt.show()

In [ ]:
df_analysis["Other_students_kind_num"] = df_analysis["Other_students_kind_and_helpful"].map({
    "Never": 0,
    "Rarely": 1,
    "Sometimes": 2,
    "Most of the time": 3,
    "Always": 4
})

In [ ]:
kind_table_pct = pd.crosstab(
    df_analysis["Other_students_kind_num"],
    df_analysis["bullied"],
    normalize="index"
)

kind_table_pct = kind_table_pct.sort_index()

labels = ["Never", "Rarely", "Sometimes", "Most", "Always"]
kind_table_pct.index = labels

kind_table_pct

In [ ]:
kind_table_pct.plot(kind="bar", figsize=(8,5))
plt.title("Bullying Rate by Students Kindness")
plt.xlabel("Other Students Kind & Helpful")
plt.ylabel("Percentage")
plt.legend(["Not Bullied", "Bullied"])
plt.show()

In [ ]:
kind_table_pct[1].plot(marker='o', figsize=(8,5))
plt.title("Bullying Probability by Students Kindness")
plt.xlabel("Other Students Kind & Helpful")
plt.ylabel("Probability")
plt.grid()
plt.show()

In [ ]:
kind_table = pd.crosstab(
    df_analysis["Other_students_kind_num"],
    df_analysis["bullied"]
)


chi2, p, dof, expected = chi2_contingency(kind_table)

print("Chi-Square Test Results:")
print("------------------------")
print(f"Chi2 Statistic: {chi2:.2f}")
print(f"P-value: {p:.5f}")
print(f"Degrees of Freedom: {dof}")

if p < 0.05:
    print("\nResult: Significant relationship (reject H0)")
else:
    print("\nResult: No significant relationship (fail to reject H0)")

In [ ]:
kind_expected_df = pd.DataFrame(
    expected,
    index=kind_table.index,
    columns=kind_table.columns
)

kind_expected_df

In [ ]:
kind_model_data = df_analysis[["Other_students_kind_num", "bullied"]].dropna()

spearmanr(
    kind_model_data["Other_students_kind_num"],
    kind_model_data["bullied"]
)

In [ ]:
X = kind_model_data[["Other_students_kind_num"]]
y = kind_model_data["bullied"]

kind_model = LogisticRegression()
kind_model.fit(X, y)

kind_model.coef_

In [ ]:
kind_model.intercept_

In [ ]:
x_vals = np.array([0,1,2,3,4])
probs = kind_model.predict_proba(x_vals.reshape(-1,1))[:,1]

plt.plot(x_vals, probs, marker='o')
plt.xticks(x_vals, ["Never","Rarely","Sometimes","Most","Always"])
plt.ylabel("Probability of Being Bullied")
plt.title("Logistic Regression Effect - Students Kindness")
plt.grid()
plt.show()

In [ ]:
df_analysis["Parents_understand_num"] = df_analysis["Parents_understand_problems"].map({
    "Never": 0,
    "Rarely": 1,
    "Sometimes": 2,
    "Most of the time": 3,
    "Always": 4
})

In [ ]:
parents_table_pct = pd.crosstab(
    df_analysis["Parents_understand_num"],
    df_analysis["bullied"],
    normalize="index"
)

parents_table_pct = parents_table_pct.sort_index()

labels = ["Never", "Rarely", "Sometimes", "Most", "Always"]
parents_table_pct.index = labels

parents_table_pct

In [ ]:
parents_table_pct.plot(kind="bar", figsize=(8,5))
plt.title("Bullying Rate by Parental Understanding")
plt.xlabel("Parents Understand Problems")
plt.ylabel("Percentage")
plt.legend(["Not Bullied", "Bullied"])
plt.show()

In [ ]:
parents_table_pct[1].plot(marker='o', figsize=(8,5))
plt.title("Bullying Probability by Parental Understanding")
plt.xlabel("Parents Understand Problems")
plt.ylabel("Probability")
plt.grid()
plt.show()

In [ ]:
parents_table = pd.crosstab(
    df_analysis["Parents_understand_num"],
    df_analysis["bullied"]
)


chi2, p, dof, expected = chi2_contingency(parents_table)

print("Chi-Square Test Results:")
print("------------------------")
print(f"Chi2 Statistic: {chi2:.2f}")
print(f"P-value: {p:.5f}")
print(f"Degrees of Freedom: {dof}")

if p < 0.05:
    print("\nResult: Significant relationship (reject H0)")
else:
    print("\nResult: No significant relationship (fail to reject H0)")

In [ ]:
parents_model_data = df_analysis[["Parents_understand_num", "bullied"]].dropna()

spearmanr(
    parents_model_data["Parents_understand_num"],
    parents_model_data["bullied"]
)

In [ ]:
X = parents_model_data[["Parents_understand_num"]]
y = parents_model_data["bullied"]

parents_model = LogisticRegression()
parents_model.fit(X, y)

parents_model.coef_

In [ ]:
x_vals = np.array([0,1,2,3,4])
probs = parents_model.predict_proba(x_vals.reshape(-1,1))[:,1]

plt.plot(x_vals, probs, marker='o')
plt.xticks(x_vals, ["Never","Rarely","Sometimes","Most","Always"])
plt.ylabel("Probability of Being Bullied")
plt.title("Logistic Regression Effect - Parental Understanding")
plt.grid()
plt.show()

In [ ]:
df_analysis["Miss_school_num"] = df_analysis["Miss_school_no_permission"].map({
    "0 days": 0,
    "1 or 2 days": 1,
    "3 to 5 days": 2,
    "6 to 9 days": 3,
    "10 or more days": 4
})

In [ ]:
miss_table_pct = pd.crosstab(
    df_analysis["Miss_school_num"],
    df_analysis["bullied"],
    normalize="index"
)

miss_table_pct = miss_table_pct.sort_index()

labels = ["0", "1-2", "3-5", "6-9", "10+"]
miss_table_pct.index = labels

miss_table_pct

In [ ]:
miss_table_pct.plot(kind="bar", figsize=(8,5))
plt.title("Bullying Rate by Missed School")
plt.xlabel("Missed School Days")
plt.ylabel("Percentage")
plt.legend(["Not Bullied", "Bullied"])
plt.show()

In [ ]:
miss_table_pct[1].plot(marker='o', figsize=(8,5))
plt.title("Bullying Probability by Missed School")
plt.xlabel("Missed School Days")
plt.ylabel("Probability")
plt.grid()
plt.show()

In [ ]:
miss_table = pd.crosstab(
    df_analysis["Miss_school_num"],
    df_analysis["bullied"]
)

chi2_contingency(miss_table)

print("Chi-Square Test Results:")
print("------------------------")
print(f"Chi2 Statistic: {chi2:.2f}")
print(f"P-value: {p:.5f}")
print(f"Degrees of Freedom: {dof}")

if p < 0.05:
    print("\nResult: Significant relationship (reject H0)")
else:
    print("\nResult: No significant relationship (fail to reject H0)")

In [ ]:
miss_model_data = df_analysis[["Miss_school_num", "bullied"]].dropna()

spearmanr(
    miss_model_data["Miss_school_num"],
    miss_model_data["bullied"]
)

In [ ]:
X = miss_model_data[["Miss_school_num"]]
y = miss_model_data["bullied"]

miss_model = LogisticRegression()
miss_model.fit(X, y)

miss_model.coef_

In [ ]:
miss_expected_df = pd.DataFrame(
    expected,
    index=miss_table.index,
    columns=miss_table.columns
)

miss_expected_df

In [ ]:
x_vals = np.array([0,1,2,3,4])
probs = miss_model.predict_proba(x_vals.reshape(-1,1))[:,1]

plt.plot(x_vals, probs, marker='o')
plt.xticks(x_vals, labels)
plt.ylabel("Probability of Being Bullied")
plt.title("Logistic Regression Effect - Missed School")
plt.grid()
plt.show()

In [ ]:
df_analysis["Age_group"] = df_analysis["Custom_Age"].map({
    "11 years old or younger": "13 or younger",
    "12 years old": "13 or younger",
    "13 years old": "13 or younger",

    "14 years old": "14-15",
    "15 years old": "14-15",

    "16 years old": "16+",
    "17 years old": "16+",
    "18 years old or older": "16+"
})

In [ ]:
age_order = ["13 or younger", "14-15", "16+"]

df_analysis["Age_group_num"] = df_analysis["Age_group"].map({
    k: i for i, k in enumerate(age_order)
})

In [ ]:
age_table_pct = pd.crosstab(
    df_analysis["Age_group"],
    df_analysis["bullied"],
    normalize="index"
)

age_table_pct

In [ ]:
age_table_pct.plot(kind="bar", figsize=(8,5))
plt.title("Bullying Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Percentage")
plt.legend(["Not Bullied", "Bullied"])
plt.show()

In [ ]:
age_table_pct[1].plot(marker='o', figsize=(8,5))
plt.title("Bullying Probability by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Probability")
plt.grid()
plt.show()

In [ ]:
age_table = pd.crosstab(
    df_analysis["Age_group"],
    df_analysis["bullied"]
)

age_table



In [ ]:
chi2, p, dof, expected = chi2_contingency(age_table)

age_expected_df = pd.DataFrame(
    expected,
    index=age_table.index,
    columns=age_table.columns
)

age_expected_df

In [ ]:
print("Chi-Square Test Results:")
print("------------------------")
print(f"Chi2 Statistic: {chi2:.2f}")
print(f"P-value: {p:.5f}")
print(f"Degrees of Freedom: {dof}")

if p < 0.05:
    print("\nResult: Significant relationship (reject H0)")
else:
    print("\nResult: No significant relationship (fail to reject H0)")

In [ ]:
age_model_data = df_analysis[["Age_group_num", "bullied"]].dropna()

spearmanr(
    age_model_data["Age_group_num"],
    age_model_data["bullied"]
)

In [ ]:
X = age_model_data[["Age_group_num"]]
y = age_model_data["bullied"]

age_model = LogisticRegression()
age_model.fit(X, y)

age_model.coef_

In [ ]:
age_model.intercept_

In [ ]:
x_vals = np.array([0,1,2])
probs = age_model.predict_proba(x_vals.reshape(-1,1))[:,1]

plt.plot(x_vals, probs, marker='o')
plt.xticks(x_vals, ["13 or younger", "14-15", "16+"])
plt.ylabel("Probability of Being Bullied")
plt.title("Logistic Regression Effect - Age")
plt.grid()
plt.show()

In [ ]:
features = [
    "Felt_lonely_num",
    "Physically_attacked_group",
    "Physical_fighting_group",
    "Other_students_kind_num",
    "Parents_understand_num",
    "Miss_school_num",
    "Close_friends_num",
    "Sex_num",
    "Age_group_num"
]

In [ ]:
model_data = df_analysis[features + ["bullied"]].dropna()

X = model_data[features]
y = model_data["bullied"]

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

In [ ]:
coef_df = pd.DataFrame({
    "feature": features,
    "coef": model.coef_[0]
}).sort_values(by="coef", key=abs, ascending=False)

coef_df

In [ ]:
coef_df["odds_ratio"] = np.exp(coef_df["coef"])
coef_df

In [ ]:
coef_df.sort_values("coef").plot(
    x="feature",
    y="coef",
    kind="barh",
    figsize=(8,5)
)

plt.title("Feature Impact on Bullying (Logistic Regression)")
plt.xlabel("Effect Size")
plt.ylabel("Feature")
plt.grid()
plt.show()

In [ ]:
summary_tests = [
    ("Felt_lonely", lonely_table, lonely_model_data["Felt_lonely_num"], lonely_model_data["bullied"]),
    ("Close_friends", friends_table, friends_model_data["Close_friends_num"], friends_model_data["bullied"]),
    ("Sex", sex_table, None, None),
    ("Physically_attacked", attacked_table, attacked_model_data["Physically_attacked_group"], attacked_model_data["bullied"]),
    ("Physical_fighting", fighting_table, fighting_model_data["Physical_fighting_group"], fighting_model_data["bullied"]),
    ("Other_students_kind", kind_table, kind_model_data["Other_students_kind_num"], kind_model_data["bullied"]),
    ("Parents_understand", parents_table, parents_model_data["Parents_understand_num"], parents_model_data["bullied"]),
    ("Miss_school", miss_table, miss_model_data["Miss_school_num"], miss_model_data["bullied"]),
    ("Age_group", age_table, age_model_data["Age_group_num"], age_model_data["bullied"]),
]

results = []

for name, table, x, y in summary_tests:
    chi2, p, dof, expected = chi2_contingency(table)

    if x is not None:
        spearman_corr, spearman_p = spearmanr(x, y)
    else:
        spearman_corr, spearman_p = None, None

    results.append({
        "feature": name,
        "chi2": chi2,
        "chi2_p_value": p,
        "dof": dof,
        "spearman_corr": spearman_corr,
        "spearman_p_value": spearman_p
    })

tests_summary_df = pd.DataFrame(results)

tests_summary_df

In [ ]:
tests_summary_df.sort_values(
    by="spearman_corr",
    key=lambda x: x.abs(),
    ascending=False
)

In [ ]:
tests_summary_df.sort_values(by="chi2", ascending=False)

In [ ]:
df_analysis["lonely_kindness_interaction"] = (
    df_analysis["Felt_lonely_num"] * df_analysis["Other_students_kind_num"]
)

features_interaction = features + ["lonely_kindness_interaction"]

model_data = df_analysis[features_interaction + ["bullied"]].dropna()

X = model_data[features_interaction]
y = model_data["bullied"]

model_interaction = LogisticRegression(max_iter=1000)
model_interaction.fit(X, y)

pd.DataFrame({
    "feature": features_interaction,
    "coef": model_interaction.coef_[0]
}).sort_values(by="coef", key=abs, ascending=False)

In [ ]:
df_analysis["lonely_parents_interaction"] = (
    df_analysis["Felt_lonely_num"] * df_analysis["Parents_understand_num"]
)

features_interaction = features + ["lonely_parents_interaction"]

model_data = df_analysis[features_interaction + ["bullied"]].dropna()
X = model_data[features_interaction]
y = model_data["bullied"]

model_lp = LogisticRegression(max_iter=1000)
model_lp.fit(X, y)

pd.DataFrame({
    "feature": features_interaction,
    "coef": model_lp.coef_[0]
}).sort_values(by="coef", key=abs, ascending=False)

In [ ]:
df_analysis["attacked_lonely_interaction"] = (
    df_analysis["Physically_attacked_group"] * df_analysis["Felt_lonely_num"]
)

features_interaction = features + ["attacked_lonely_interaction"]

model_data = df_analysis[features_interaction + ["bullied"]].dropna()
X = model_data[features_interaction]
y = model_data["bullied"]

model_al = LogisticRegression(max_iter=1000)
model_al.fit(X, y)

pd.DataFrame({
    "feature": features_interaction,
    "coef": model_al.coef_[0]
}).sort_values(by="coef", key=abs, ascending=False)